# Test Augmented Tables
Validate the BattingAugmented and PitchingAugmented CSVs from `./`.

In [1]:
import pandas as pd

batting = pd.read_csv('./Batting.csv')
pitching = pd.read_csv('./Pitching.csv')
people = pd.read_csv('../raw_data/People.csv')

print(f'Batting: {len(batting)} rows, {batting["WAR"].notna().sum()} with WAR')
print(f'Pitching: {len(pitching)} rows, {pitching["WAR"].notna().sum()} with WAR')

Batting: 25132 rows, 25132 with WAR
Pitching: 20455 rows, 20455 with WAR


## Spot-check: Aaron Judge (multi-year)

In [2]:
judge = batting[batting['playerID'] == 'judgeaa01'][['playerID', 'yearID', 'teamID', 'G', 'AB', 'HR', 'RBI', 'WAR']]
judge.sort_values('yearID')

,playerID,yearID,teamID,G,AB,HR,RBI,WAR
11615,judgeaa01,2016,NYA,27,84,4,10.0,-0.3
11616,judgeaa01,2017,NYA,155,542,52,114.0,8.1
11617,judgeaa01,2018,NYA,112,413,27,67.0,6.0
11618,judgeaa01,2019,NYA,102,378,27,55.0,5.6
11619,judgeaa01,2020,NYA,28,101,9,22.0,1.1
11620,judgeaa01,2021,NYA,148,550,39,98.0,5.9
11621,judgeaa01,2022,NYA,157,570,62,131.0,10.8
11622,judgeaa01,2023,NYA,106,367,37,75.0,4.6
11623,judgeaa01,2024,NYA,158,559,58,144.0,10.9
11624,judgeaa01,2025,NYA,152,541,53,114.0,9.7


## Spot-check: Shohei Ohtani (batting + pitching)

In [3]:
ohtani_bat = batting[batting['playerID'] == 'ohtansh01'][['playerID', 'yearID', 'teamID', 'G', 'AB', 'HR', 'WAR']]
print('Ohtani Batting:')
display(ohtani_bat.sort_values('yearID'))

ohtani_pit = pitching[pitching['playerID'] == 'ohtansh01'][['playerID', 'yearID', 'teamID', 'G', 'W', 'L', 'ERA', 'WAR']]
print('Ohtani Pitching:')
display(ohtani_pit.sort_values('yearID'))

Ohtani Batting:


,playerID,yearID,teamID,G,AB,HR,WAR
16597,ohtansh01,2018,LAA,114,326,22,2.7
16598,ohtansh01,2019,LAA,106,384,18,2.4
16599,ohtansh01,2020,LAA,46,153,7,0.0
16600,ohtansh01,2021,LAA,158,537,46,4.9
16601,ohtansh01,2022,LAA,157,586,34,3.4
16602,ohtansh01,2023,LAA,135,497,44,6.1
16603,ohtansh01,2024,LAN,159,636,54,9.0
16604,ohtansh01,2025,LAN,158,611,55,6.6


Ohtani Pitching:


,playerID,yearID,teamID,G,W,L,ERA,WAR
13282,ohtansh01,2018,LAA,10,4,2,3.31,1.3
13283,ohtansh01,2020,LAA,2,0,1,37.80,-0.4
13284,ohtansh01,2021,LAA,23,9,2,3.18,4.1
13285,ohtansh01,2022,LAA,28,15,9,2.33,6.3
13286,ohtansh01,2023,LAA,23,10,5,3.14,3.8
13287,ohtansh01,2025,LAN,14,1,1,2.87,1.1


## Multi-team player check
Players who switched teams mid-season should have separate rows with WAR for each team.

In [4]:
# Find players with multiple stints in a single year
multi = batting.groupby(['playerID', 'yearID']).filter(lambda g: len(g) > 1)
multi_with_war = multi[multi['WAR'].notna()]
print(f'Multi-team rows: {len(multi)}, with WAR: {len(multi_with_war)}')

# Show a specific example
example_player = multi_with_war.groupby('playerID').first().index[0]
example = multi[multi['playerID'] == example_player][['playerID', 'yearID', 'teamID', 'stint', 'G', 'AB', 'HR', 'WAR']]
print(f'\nExample: {example_player}')
display(example.sort_values(['yearID', 'stint']))

Multi-team rows: 3481, with WAR: 3481

Example: abernbr01


,playerID,yearID,teamID,stint,G,AB,HR,WAR
22,abernbr01,2003,TBA,1,2,7,0,0.0
23,abernbr01,2003,KCA,2,10,27,0,-0.8


## Rows missing WAR
Check what's missing — likely low-AB players not in BBRef WAR tables.

In [5]:
missing = batting[batting['WAR'].isna()]
print(f'Missing WAR: {len(missing)} rows')
print(f'Mean AB for missing: {missing["AB"].mean():.1f}')
print(f'Mean AB for present: {batting[batting["WAR"].notna()]["AB"].mean():.1f}')
print(f'\nAB distribution of missing WAR rows:')
missing['AB'].describe()

Missing WAR: 0 rows
Mean AB for missing: nan
Mean AB for present: 167.1

AB distribution of missing WAR rows:


count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: AB, dtype: float64

## WAR leaders by year

In [6]:
bat_leaders = batting.dropna(subset=['WAR']).merge(
    people[['playerID', 'nameFirst', 'nameLast']], on='playerID'
)
bat_leaders['Name'] = bat_leaders['nameFirst'] + ' ' + bat_leaders['nameLast']

# Top 5 batting WAR per year
for year in sorted(batting['yearID'].unique()):
    top = bat_leaders[bat_leaders['yearID'] == year].nlargest(5, 'WAR')[['Name', 'teamID', 'WAR']]
    print(f'\n--- {year} Batting WAR Leaders ---')
    print(top.to_string(index=False))


--- 2000 Batting WAR Leaders ---
          Name teamID  WAR
Alex Rodriguez    SEA 10.4
   Todd Helton    COL  8.9
  Darin Erstad    ANA  8.4
  Andruw Jones    ATL  8.2
  Jason Giambi    OAK  7.8

--- 2001 Batting WAR Leaders ---
          Name teamID  WAR
   Barry Bonds    SFN 11.9
    Sammy Sosa    CHN 10.3
  Jason Giambi    OAK  9.2
    Bret Boone    SEA  8.8
Alex Rodriguez    TEX  8.3

--- 2002 Batting WAR Leaders ---
             Name teamID  WAR
      Barry Bonds    SFN 11.8
   Alex Rodriguez    TEX  8.8
        Jim Thome    CLE  7.4
     Jason Giambi    NYA  7.1
Vladimir Guerrero    MON  7.1

--- 2003 Batting WAR Leaders ---
          Name teamID  WAR
   Barry Bonds    SFN  9.2
 Albert Pujols    SLN  8.7
Alex Rodriguez    TEX  8.4
  Marcus Giles    ATL  7.9
    Javy Lopez    ATL  6.8

--- 2004 Batting WAR Leaders ---
         Name teamID  WAR
  Barry Bonds    SFN 10.6
Adrian Beltre    LAN  9.6
  Scott Rolen    SLN  9.2
Ichiro Suzuki    SEA  9.2
Albert Pujols    SLN  8.5

--- 200